# 01 — Raw Data Audit

**Project:** TTC Transit Reliability Analytics
**Stage:** 1 — Raw Data Audit
**Goal of this notebook:** before cleaning a single row, understand exactly what we have on disk — file by file, column by column — and decide whether the three modes (subway, bus, streetcar) can be standardised into one combined fact table.

This notebook is the *narrative* part of Stage 1. It should be readable top-to-bottom by someone who has never seen the project. Each section opens with a plain-language explanation of *what* we are checking and *why* it matters; the code that follows just runs the check and prints the answer.

### Why we audit before cleaning

Cleaning code makes assumptions about the data ("the date is in column X", "delay codes are 3-digit strings"). If those assumptions are wrong, the cleaning step either crashes or — much worse — silently produces bad numbers. The audit's job is to *find and write down the real shape of the data* so the cleaning step is built on facts, not guesses.

### Scope reminder

We deliberately use 2022 → present (see [`docs/decisions.md`](../docs/decisions.md), Decision 1). The audit covers four full years of XLSX (2022, 2023, 2024) plus the 2025-onward CSV per mode.

### Specific things we are looking for

Per CLAUDE.md and [`docs/project_plan.md`](../docs/project_plan.md), the audit must confirm:

1. **Files & shape** — every raw file's row count, columns, dtypes, first rows.
2. **Date range** — earliest and latest date per file.
3. **Missing values** — count and percentage per column.
4. **Excel quirks** — are workbooks split into monthly tabs? Did codes lose leading zeros? Did dates parse as numeric serials?
5. **Schema drift** — do columns change between the XLSX years and the 2025 CSV?
6. **Code-description join** — every delay code in the event files should appear in that mode's `Code Descriptions.csv`.
7. **Combined-fact-table feasibility** — which columns are common across modes? Which are mode-specific?

A concise summary of every answer is collected at the bottom of the notebook.


## 0. Setup

We use only `pandas`, `pathlib`, `openpyxl` (engine for `.xlsx`), and `re`. Nothing fancy — this is a structure-and-quality audit, not analysis.

`pd.set_option(...)` lines just make the printed tables wider/longer so you can read columns without truncation when scrolling the notebook.


In [1]:
import re
from pathlib import Path

import pandas as pd
import openpyxl

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

# Project paths. The notebook lives in notebooks/, so the project root is one up.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = PROJECT_ROOT / "data" / "raw"
print("Project root:", PROJECT_ROOT)
print("Raw data dir:", RAW)
print("Raw exists? ", RAW.exists())


Project root: C:\Users\soham\Desktop\Coding Stuff\TTC Transit Delay & Reliability Analysis
Raw data dir: C:\Users\soham\Desktop\Coding Stuff\TTC Transit Delay & Reliability Analysis\data\raw
Raw exists?  True


## 1. File inventory (programmatic)

We already wrote [`docs/data_inventory.md`](../docs/data_inventory.md) by hand. This cell confirms the same files are on disk programmatically — so future runs of the notebook fail loudly if a file goes missing, rather than silently skipping it.


In [2]:
mode_dirs = {m: RAW / m for m in ["subway", "bus", "streetcar"]}
for mode, d in mode_dirs.items():
    print(f"\n--- {mode} ({d.name}) ---")
    for f in sorted(d.iterdir()):
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name:55s}  {size_kb:>10.1f} KB")



--- subway (subway) ---
  Code Descriptions.csv                                           5.5 KB
  TTC Subway Delay Data since 2025.csv                         1889.9 KB
  ttc-subway-delay-codes.xlsx                                    19.7 KB
  ttc-subway-delay-data-2022.xlsx                              1061.6 KB
  ttc-subway-delay-data-2023.xlsx                              1208.1 KB
  ttc-subway-delay-data-2024.xlsx                              1390.2 KB
  ttc-subway-delay-data-readme.xlsx                              12.4 KB

--- bus (bus) ---
  Code Descriptions.csv                                           1.3 KB
  TTC Bus Delay Data since 2025.csv                            6309.7 KB
  ttc-bus-delay-data-2022.xlsx                                 3407.9 KB
  ttc-bus-delay-data-2023.xlsx                                 3308.9 KB
  ttc-bus-delay-data-2024.xlsx                                 3506.2 KB
  ttc-bus-delay-data-readme.xlsx                                 12.5 KB

--- st

**What to check:** every mode should have:

- one `... since 2025.csv`
- three `...-2022.xlsx`, `...-2023.xlsx`, `...-2024.xlsx`
- one `Code Descriptions.csv`
- one `...-readme.xlsx`

Subway has one extra file (`ttc-subway-delay-codes.xlsx`) that we intentionally do **not** use — see Decision 5 in `docs/decisions.md`.


## 2. Helper functions

Three small helpers, defined once so each per-mode section reads cleanly:

- `excel_sheet_info(path)` — opens an XLSX in read-only mode and returns just the sheet names. This is how we detect "one tab per month" workbooks before reading them.
- `read_excel_stacked(path)` — reads *every* sheet, concatenates them, and tags each row with which sheet it came from. This is what we'll use for any XLSX that has multiple sheets.
- `audit_event_frame(df, label)` — prints the same 5 things for every event dataset: shape, columns, dtypes, date range, missing-value summary. Uniform output makes cross-file comparison much easier.

Defining helpers here also keeps the actual audit cells short and readable — each one is one or two function calls, not 30 lines of repeated boilerplate.


In [3]:
def excel_sheet_info(path: Path) -> list[str]:
    """Return the list of sheet names in an XLSX without loading data."""
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    try:
        return list(wb.sheetnames)
    finally:
        wb.close()


def read_excel_stacked(path: Path) -> pd.DataFrame:
    """Read every sheet in an XLSX, concatenate, and tag rows with sheet name.

    Why: TTC has historically shipped one Excel tab per month. If we read only
    the first sheet, we silently lose 11/12 of the year. Reading them all and
    stacking gives us a single annual DataFrame plus a `_sheet` column for
    traceability.
    """
    sheets = pd.read_excel(path, sheet_name=None)  # dict[sheet_name, DataFrame]
    parts = []
    for name, part in sheets.items():
        part = part.copy()
        part["_sheet"] = name
        parts.append(part)
    return pd.concat(parts, ignore_index=True, sort=False)


def find_date_column(df: pd.DataFrame) -> str | None:
    """Best-effort detection of the date column (case-insensitive 'date' match)."""
    for c in df.columns:
        if "date" in str(c).lower():
            return c
    return None


def audit_event_frame(df: pd.DataFrame, label: str) -> dict:
    """Print and return a uniform audit summary for one event DataFrame."""
    print(f"\n========== {label} ==========")
    print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print(f"Columns: {list(df.columns)}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nFirst 3 rows:")
    print(df.head(3))

    summary = {"label": label, "rows": int(df.shape[0]), "cols": int(df.shape[1]),
               "columns": list(df.columns)}

    date_col = find_date_column(df)
    summary["date_column"] = date_col
    if date_col is not None:
        parsed = pd.to_datetime(df[date_col], errors="coerce")
        nat = int(parsed.isna().sum())
        summary["date_min"] = str(parsed.min())
        summary["date_max"] = str(parsed.max())
        summary["date_unparseable"] = nat
        print(f"\nDate column '{date_col}': "
              f"min={parsed.min()}, max={parsed.max()}, unparseable={nat}")

    miss = df.isna().sum()
    miss_pct = (miss / len(df) * 100).round(2)
    miss_df = pd.DataFrame({"missing": miss, "missing_pct": miss_pct})
    miss_df = miss_df[miss_df["missing"] > 0].sort_values("missing", ascending=False)
    summary["missing"] = miss_df.to_dict(orient="index")
    if miss_df.empty:
        print("\nMissing values: none")
    else:
        print(f"\nMissing values (columns with at least one NaN):")
        print(miss_df)

    return summary


# Collected summaries, indexed by label, used by the final section.
AUDIT = {}


## 3. Subway audit

We start with subway because TTC publishes the most thorough metadata for it, and any column-naming convention we discover here gives us a baseline to compare bus and streetcar against.

### 3.1 Sheet structure of the XLSX files

Question: are the 2022/2023/2024 workbooks single-sheet (whole year in one tab) or multi-sheet (one tab per month, ~12 sheets)? The answer changes how `read_excel` must be called.


In [4]:
subway_xlsx = {
    2022: RAW / "subway" / "ttc-subway-delay-data-2022.xlsx",
    2023: RAW / "subway" / "ttc-subway-delay-data-2023.xlsx",
    2024: RAW / "subway" / "ttc-subway-delay-data-2024.xlsx",
}
for year, path in subway_xlsx.items():
    sheets = excel_sheet_info(path)
    print(f"{year}: {len(sheets)} sheet(s) -> {sheets}")


2022: 1 sheet(s) -> ['2022']
2023: 1 sheet(s) -> ['2023']
2024: 1 sheet(s) -> ['Subway']


**Interpretation:** the printed sheet count tells us whether to read one sheet or all of them. If there are 12 sheets named like months, we need `read_excel_stacked`. If there is one sheet, a single `pd.read_excel(...)` would suffice — but using the same `read_excel_stacked` helper anyway costs nothing and keeps the pipeline uniform.

### 3.2 Stack all sheets per year, then audit

We read every sheet, concatenate them, and run the uniform audit. The `_sheet` column added by `read_excel_stacked` lets us verify that all months are present in the stacked frame.


In [5]:
for year, path in subway_xlsx.items():
    df = read_excel_stacked(path)
    AUDIT[f"subway_{year}"] = audit_event_frame(df, f"subway {year} (XLSX, stacked)")
    print(f"\nSheet origin counts (should sum to total rows):")
    print(df["_sheet"].value_counts().sort_index())



========== subway 2022 (XLSX, stacked) ==========
Shape: 19,895 rows x 11 columns
Columns: ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle', '_sheet']

Dtypes:
Date         datetime64[us]
Time                    str
Day                     str
Station                 str
Code                    str
Min Delay             int64
Min Gap               int64
Bound                   str
Line                    str
Vehicle               int64
_sheet                  str
dtype: object

First 3 rows:
        Date   Time       Day                 Station  Code  Min Delay  Min Gap Bound Line  Vehicle _sheet
0 2022-01-01  15:59  Saturday   LAWRENCE EAST STATION  SRDP          0        0     N  SRT     3023   2022
1 2022-01-01  02:23  Saturday      SPADINA BD STATION  MUIS          0        0   NaN   BD        0   2022
2 2022-01-01  22:00  Saturday  KENNEDY SRT STATION TO   MRO          0        0   NaN  SRT        0   2022

Date column 'Date': min=2022


========== subway 2023 (XLSX, stacked) ==========
Shape: 22,949 rows x 11 columns
Columns: ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle', '_sheet']

Dtypes:
Date         datetime64[us]
Time                    str
Day                     str
Station                 str
Code                    str
Min Delay             int64
Min Gap               int64
Bound                   str
Line                    str
Vehicle               int64
_sheet                  str
dtype: object

First 3 rows:
        Date   Time     Day          Station   Code  Min Delay  Min Gap Bound Line  Vehicle _sheet
0 2023-01-01  02:22  Sunday   MUSEUM STATION  MUPAA          3        9     S   YU     5931   2023
1 2023-01-01  02:30  Sunday  KIPLING STATION   MUIS          0        0     E   BD     5341   2023
2 2023-01-01  02:33  Sunday   WARDEN STATION    SUO          0        0     W   BD        0   2023

Date column 'Date': min=2023-01-01 00:00:00, max=2023-12-31 


========== subway 2024 (XLSX, stacked) ==========
Shape: 26,467 rows x 11 columns
Columns: ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle', '_sheet']

Dtypes:
Date         datetime64[us]
Time                    str
Day                     str
Station                 str
Code                    str
Min Delay             int64
Min Gap               int64
Bound                   str
Line                    str
Vehicle               int64
_sheet                  str
dtype: object

First 3 rows:
        Date   Time     Day           Station   Code  Min Delay  Min Gap Bound Line  Vehicle  _sheet
0 2024-01-01  02:00  Monday  SHEPPARD STATION    MUI          0        0     N   YU     5491  Subway
1 2024-01-01  02:00  Monday    DUNDAS STATION   MUIS          0        0     N   YU        0  Subway
2 2024-01-01  02:08  Monday    DUNDAS STATION  MUPAA          4       10     N   YU     6051  Subway

Date column 'Date': min=2024-01-01 00:00:00, max=202

### 3.3 Subway 2025-onward CSV

In [6]:
subway_csv_path = RAW / "subway" / "TTC Subway Delay Data since 2025.csv"
subway_csv = pd.read_csv(subway_csv_path)
AUDIT["subway_2025plus"] = audit_event_frame(subway_csv, "subway 2025+ (CSV)")



========== subway 2025+ (CSV) ==========
Shape: 28,191 rows x 11 columns
Columns: ['_id', 'Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']

Dtypes:
_id          int64
Date           str
Time           str
Day            str
Station        str
Code           str
Min Delay    int64
Min Gap      int64
Bound          str
Line           str
Vehicle      int64
dtype: object

First 3 rows:
   _id        Date   Time        Day            Station   Code  Min Delay  Min Gap Bound Line  Vehicle
0    1  2025-01-01  02:10  Wednesday   BATHURST STATION  MUSAN          5        9     E   BD     5227
1    2  2025-01-01  02:30  Wednesday     DUNDAS STATION  MUIRS          0        0   NaN   YU        0
2    3  2025-01-01  02:32  Wednesday  BROADVIEW STATION  PUMST          0        0     E   BD        0

Date column 'Date': min=2025-01-01 00:00:00, max=2026-01-31 00:00:00, unparseable=0

Missing values (columns with at least one NaN):
       missing  missin

### 3.4 Subway code-description lookup

This is what we'll eventually `JOIN` the event table to so each row gains a human-readable description and category.


In [7]:
subway_codes_path = RAW / "subway" / "Code Descriptions.csv"
subway_codes = pd.read_csv(subway_codes_path)
print(f"Shape: {subway_codes.shape}")
print(f"Columns: {list(subway_codes.columns)}")
print(f"\nFirst 10 rows:")
print(subway_codes.head(10))
print(f"\nDuplicate code rows: {subway_codes.duplicated(subset=subway_codes.columns[0]).sum()}")
AUDIT["subway_codes"] = {
    "rows": int(subway_codes.shape[0]),
    "columns": list(subway_codes.columns),
    "unique_codes": int(subway_codes.iloc[:, 0].nunique()),
}


Shape: (140, 3)
Columns: ['_id', 'CODE', 'DESCRIPTION']

First 10 rows:
   _id   CODE                                                  DESCRIPTION
0    1   EUAC                                             AIR CONDITIONING
1    2   EUAL                                          ALTERNATING CURRENT
2    3  EUATC                                           ATC RC&S EQUIPMENT
3    4   EUBK                                                       BRAKES
4    5   EUBO                                                         BODY
5    6   EUCA                                               COMPRESSED AIR
6    7   EUCC                                                  CAM CONTROL
7    8   EUCD  RC&S CONSEQUENTIAL DELAY (SECOND DELAY SAME FAULT â NO...
8    9   EUCH                                              CHOPPER CONTROL
9   10   EUCO                                                     COUPLERS

Duplicate code rows: 0


## 4. Bus audit

Same checks as subway. We expect different columns (bus has routes/directions instead of stations/lines), but the audit structure is identical.


In [8]:
bus_xlsx = {
    2022: RAW / "bus" / "ttc-bus-delay-data-2022.xlsx",
    2023: RAW / "bus" / "ttc-bus-delay-data-2023.xlsx",
    2024: RAW / "bus" / "ttc-bus-delay-data-2024.xlsx",
}
for year, path in bus_xlsx.items():
    sheets = excel_sheet_info(path)
    print(f"{year}: {len(sheets)} sheet(s) -> {sheets}")


2022: 1 sheet(s) -> ['2022']
2023: 1 sheet(s) -> ['2023']


2024: 1 sheet(s) -> ['2024']


In [9]:
for year, path in bus_xlsx.items():
    df = read_excel_stacked(path)
    AUDIT[f"bus_{year}"] = audit_event_frame(df, f"bus {year} (XLSX, stacked)")
    print(f"\nSheet origin counts:")
    print(df["_sheet"].value_counts().sort_index())



========== bus 2022 (XLSX, stacked) ==========
Shape: 58,707 rows x 11 columns
Columns: ['Date', 'Route', 'Time', 'Day', 'Location', 'Incident', 'Min Delay', 'Min Gap', 'Direction', 'Vehicle', '_sheet']

Dtypes:
Date         datetime64[us]
Route                object
Time                    str
Day                     str
Location                str
Incident                str
Min Delay             int64
Min Gap               int64
Direction               str
Vehicle               int64
_sheet                  str
dtype: object

First 3 rows:
        Date Route   Time       Day                Location               Incident  Min Delay  Min Gap Direction  Vehicle _sheet
0 2022-01-01   320  02:00  Saturday        YONGE AND DUNDAS          General Delay          0        0       NaN     8531   2022
1 2022-01-01   325  02:00  Saturday  OVERLEA AND THORCLIFFE              Diversion        131      161         W     8658   2022
2 2022-01-01   320  02:00  Saturday       YONGE AND STEELES  Op


========== bus 2023 (XLSX, stacked) ==========
Shape: 56,207 rows x 11 columns
Columns: ['Date', 'Route', 'Time', 'Day', 'Location', 'Incident', 'Min Delay', 'Min Gap', 'Direction', 'Vehicle', '_sheet']

Dtypes:
Date         datetime64[us]
Route                object
Time                    str
Day                     str
Location                str
Incident                str
Min Delay             int64
Min Gap               int64
Direction               str
Vehicle               int64
_sheet                  str
dtype: object

First 3 rows:
        Date Route   Time     Day               Location               Incident  Min Delay  Min Gap Direction  Vehicle _sheet
0 2023-01-01    91  02:30  Sunday  WOODBINE AND MORTIMER              Diversion         81      111       NaN     8772   2023
1 2023-01-01    69  02:34  Sunday         WARDEN STATION               Security         22       44         S     8407   2023
2 2023-01-01    35  03:06  Sunday           JANE STATION  Cleaning - Uns


========== bus 2024 (XLSX, stacked) ==========
Shape: 59,643 rows x 11 columns
Columns: ['Date', 'Route', 'Time', 'Day', 'Location', 'Incident', 'Min Delay', 'Min Gap', 'Direction', 'Vehicle', '_sheet']

Dtypes:
Date         datetime64[us]
Route                object
Time                    str
Day                     str
Location                str
Incident                str
Min Delay             int64
Min Gap               int64
Direction               str
Vehicle               int64
_sheet                  str
dtype: object

First 3 rows:
        Date Route   Time     Day            Location       Incident  Min Delay  Min Gap Direction  Vehicle _sheet
0 2024-01-01    89  02:08  Monday  KEELE AND GLENLAKE         Vision         10       20         N     7107   2024
1 2024-01-01    39  02:30  Monday       FINCH STATION  General Delay         20       40       NaN     8914   2024
2 2024-01-01   300  03:13  Monday   BLOOR AND MANNING  General Delay          0        0       NaN     85

In [10]:
bus_csv_path = RAW / "bus" / "TTC Bus Delay Data since 2025.csv"
bus_csv = pd.read_csv(bus_csv_path)
AUDIT["bus_2025plus"] = audit_event_frame(bus_csv, "bus 2025+ (CSV)")



========== bus 2025+ (CSV) ==========
Shape: 69,037 rows x 11 columns
Columns: ['_id', 'Date', 'Line', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Vehicle']

Dtypes:
_id          int64
Date           str
Line           str
Time           str
Day            str
Station        str
Code           str
Min Delay    int64
Min Gap      int64
Bound          str
Vehicle      int64
dtype: object

First 3 rows:
   _id                 Date              Line   Time        Day            Station   Code  Min Delay  Min Gap Bound  Vehicle
0    1  2025-01-01T00:00:00  102 MARKHAM ROAD  02:15  Wednesday     WARDEN STATION  MFESA         20       40     N     3442
1    2  2025-01-01T00:00:00     65 PARLIAMENT  02:15  Wednesday    KIPLING STATION   MFUS          0        0   NaN        0
2    3  2025-01-01T00:00:00           64 MAIN  02:40  Wednesday  BROADVIEW STATION   MFUI          0        0   NaN     8546

Date column 'Date': min=2025-01-01 00:00:00, max=2026-01-31 00:00:00, 

In [11]:
bus_codes_path = RAW / "bus" / "Code Descriptions.csv"
bus_codes = pd.read_csv(bus_codes_path)
print(f"Shape: {bus_codes.shape}")
print(f"Columns: {list(bus_codes.columns)}")
print(bus_codes.head(10))
print(f"\nDuplicate code rows: {bus_codes.duplicated(subset=bus_codes.columns[0]).sum()}")
AUDIT["bus_codes"] = {
    "rows": int(bus_codes.shape[0]),
    "columns": list(bus_codes.columns),
    "unique_codes": int(bus_codes.iloc[:, 0].nunique()),
}


Shape: (46, 3)
Columns: ['_id', 'CODE', 'DESCRIPTION']
   _id   CODE       DESCRIPTION
0    1    EFB              BODY
1    2  EFCAN      CANCELLATION
2    3    EFD             DOORS
3    4   EFDB       DISC BRAKES
4    5   EFHV      HIGH VOLTAGE
5    6  EFHVA  HEAT / VENT / AC
6    7   EFLV       LOW VOLTAGE
7    8    EFO             OTHER
8    9    EFP        PROPULSION
9   10   EFRA       RAMP ISSUES

Duplicate code rows: 0


## 5. Streetcar audit

In [12]:
streetcar_xlsx = {
    2022: RAW / "streetcar" / "ttc-streetcar-delay-data-2022.xlsx",
    2023: RAW / "streetcar" / "ttc-streetcar-delay-data-2023.xlsx",
    2024: RAW / "streetcar" / "ttc-streetcar-delay-data-2024.xlsx",
}
for year, path in streetcar_xlsx.items():
    sheets = excel_sheet_info(path)
    print(f"{year}: {len(sheets)} sheet(s) -> {sheets}")


2022: 1 sheet(s) -> ['2022']
2023: 1 sheet(s) -> ['2023']
2024: 1 sheet(s) -> ['Data']


In [13]:
for year, path in streetcar_xlsx.items():
    df = read_excel_stacked(path)
    AUDIT[f"streetcar_{year}"] = audit_event_frame(df, f"streetcar {year} (XLSX, stacked)")
    print(f"\nSheet origin counts:")
    print(df["_sheet"].value_counts().sort_index())



========== streetcar 2022 (XLSX, stacked) ==========
Shape: 17,655 rows x 11 columns
Columns: ['Date', 'Line', 'Time', 'Day', 'Location', 'Incident', 'Min Delay', 'Min Gap', 'Bound', 'Vehicle', '_sheet']

Dtypes:
Date         datetime64[us]
Line                 object
Time                    str
Day                     str
Location                str
Incident                str
Min Delay             int64
Min Gap               int64
Bound                   str
Vehicle               int64
_sheet                  str
dtype: object

First 3 rows:
        Date Line   Time       Day           Location                  Incident  Min Delay  Min Gap Bound  Vehicle _sheet
0 2022-01-01  504  02:21  Saturday  BROADVIEW STATION  Collision - TTC Involved         30       60     E     8333   2022
1 2022-01-01  501  03:22  Saturday  718 QUEEN ST EAST                Operations         16       35     W     8068   2022
2 2022-01-01  504  03:28  Saturday  BROADVIEW STATION                Operations    


========== streetcar 2023 (XLSX, stacked) ==========
Shape: 14,413 rows x 11 columns
Columns: ['Date', 'Line', 'Time', 'Day', 'Location', 'Incident', 'Min Delay', 'Min Gap', 'Bound', 'Vehicle', '_sheet']

Dtypes:
Date         datetime64[us]
Line                 object
Time                    str
Day                     str
Location                str
Incident                str
Min Delay             int64
Min Gap               int64
Bound                   str
Vehicle               int64
_sheet                  str
dtype: object

First 3 rows:
        Date Line   Time     Day                Location               Incident  Min Delay  Min Gap Bound  Vehicle _sheet
0 2023-01-01  509  02:37  Sunday  QUEENS QUAY AND SPADIN             Operations          0        0     E     4403   2023
1 2023-01-01  505  02:40  Sunday   BROADVIEW AND GERRARD                Held By         15       25     W     4460   2023
2 2023-01-01  504  02:52  Sunday       KING AND BATHURST  Cleaning - Unsanitary    


========== streetcar 2024 (XLSX, stacked) ==========
Shape: 14,206 rows x 11 columns
Columns: ['Date', 'Line', 'Time', 'Day', 'Location', 'Incident', 'Min Delay', 'Min Gap', 'Bound', 'Vehicle', '_sheet']

Dtypes:
Date         datetime64[us]
Line                 object
Time                    str
Day                     str
Location                str
Incident                str
Min Delay             int64
Min Gap               int64
Bound                   str
Vehicle               int64
_sheet                  str
dtype: object

First 3 rows:
        Date Line   Time     Day                Location            Incident  Min Delay  Min Gap Bound  Vehicle _sheet
0 2024-01-01  505  02:45  Monday       DUNDAS AND MCCAUL            Security         10       20     W     4416   Data
1 2024-01-01  505  03:06  Monday   COLLEGE AND GLADSTONE  Emergency Services         52       72     E     4461   Data
2 2024-01-01  503  03:21  Monday  PARLIAMENT AND SHUTTER            Security          0     

In [14]:
streetcar_csv_path = RAW / "streetcar" / "TTC Streetcar Delay Data since 2025.csv"
streetcar_csv = pd.read_csv(streetcar_csv_path)
AUDIT["streetcar_2025plus"] = audit_event_frame(streetcar_csv, "streetcar 2025+ (CSV)")



========== streetcar 2025+ (CSV) ==========
Shape: 15,785 rows x 11 columns
Columns: ['_id', 'Date', 'Line', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Vehicle']

Dtypes:
_id          int64
Date           str
Line           str
Time           str
Day            str
Station        str
Code           str
Min Delay    int64
Min Gap      int64
Bound          str
Vehicle      int64
dtype: object

First 3 rows:
   _id        Date         Line   Time        Day              Station   Code  Min Delay  Min Gap Bound  Vehicle
0    1  2025-01-01     504 KING  02:10  Wednesday  KING AND PARLIAMENT  MTSAN         10       20     W     4569
1    2  2025-01-01  506 CARLTON  02:50  Wednesday    COLLEGE AND HURON  MTAFR         34       49     E     4480
2    3  2025-01-01     504 KING  03:11  Wednesday  KING AND QUEEN EAST   MTIE         15       30     E     4629

Date column 'Date': min=2025-01-01 00:00:00, max=2026-01-31 00:00:00, unparseable=0

Missing values (columns wit

In [15]:
streetcar_codes_path = RAW / "streetcar" / "Code Descriptions.csv"
streetcar_codes = pd.read_csv(streetcar_codes_path)
print(f"Shape: {streetcar_codes.shape}")
print(f"Columns: {list(streetcar_codes.columns)}")
print(streetcar_codes.head(10))
print(f"\nDuplicate code rows: {streetcar_codes.duplicated(subset=streetcar_codes.columns[0]).sum()}")
AUDIT["streetcar_codes"] = {
    "rows": int(streetcar_codes.shape[0]),
    "columns": list(streetcar_codes.columns),
    "unique_codes": int(streetcar_codes.iloc[:, 0].nunique()),
}


Shape: (85, 3)
Columns: ['_id', 'CODE', 'DESCRIPTION']
   _id  CODE                                           DESCRIPTION
0    1  ETAC                                                  HVAC
1    2  ETAR                                   ARTICULATION (ALRV)
2    3  ETAX                                 AUXILARY POWER SUPPLY
3    4  ETBO                                                  BODY
4    5  ETCA                                        COMPRESSED AIR
5    6  ETCE  COMMUNICATION EQUIPMENT (INCLUDES STOP ANNOUNCEMENT)
6    7  ETCH                                  CHOPPER (PROPULSION)
7    8  ETCM                                        CAR MONITORING
8    9  ETDB                                           DISC BRAKES
9   10  ETDO                                       PASSENGER DOORS

Duplicate code rows: 0


## 6. Cross-cutting integrity checks

These three checks apply to every mode and every year, so they sit in their own section rather than being repeated inside each mode block.

### 6.1 Schema drift

We compare the columns of each year's file against the others **within the same mode**. The output is one table per mode showing which columns appear in which year. Any column with a blank in some year is a drift point we must handle during cleaning.


In [16]:
def drift_table(per_year: dict[str, list[str]]) -> pd.DataFrame:
    all_cols = sorted({c for cols in per_year.values() for c in cols if c != "_sheet"})
    rows = {label: [c in cols for c in all_cols] for label, cols in per_year.items()}
    return pd.DataFrame(rows, index=all_cols).replace({True: "x", False: ""})


for mode in ["subway", "bus", "streetcar"]:
    per_year = {
        "2022": AUDIT[f"{mode}_2022"]["columns"],
        "2023": AUDIT[f"{mode}_2023"]["columns"],
        "2024": AUDIT[f"{mode}_2024"]["columns"],
        "2025+": AUDIT[f"{mode}_2025plus"]["columns"],
    }
    print(f"\n--- {mode.upper()} schema drift ---")
    print(drift_table(per_year))



--- SUBWAY schema drift ---
          2022 2023 2024 2025+
Bound        x    x    x     x
Code         x    x    x     x
Date         x    x    x     x
Day          x    x    x     x
Line         x    x    x     x
Min Delay    x    x    x     x
Min Gap      x    x    x     x
Station      x    x    x     x
Time         x    x    x     x
Vehicle      x    x    x     x
_id                          x

--- BUS schema drift ---
          2022 2023 2024 2025+
Bound                        x
Code                         x
Date         x    x    x     x
Day          x    x    x     x
Direction    x    x    x      
Incident     x    x    x      
Line                         x
Location     x    x    x      
Min Delay    x    x    x     x
Min Gap      x    x    x     x
Route        x    x    x      
Station                      x
Time         x    x    x     x
Vehicle      x    x    x     x
_id                          x

--- STREETCAR schema drift ---
          2022 2023 2024 2025+
Bound        x

### 6.2 Delay-code integrity (leading zeros & join feasibility)

Excel happily turns the string `"0123"` into the integer `123`. If that happens, the event-file code values won't match the lookup-table values and the join would silently drop rows.

For each mode, we identify the code column in the event data, take a sample of unique codes, and check whether they exist in that mode's `Code Descriptions.csv`. We also note whether the codes are being read as strings or integers.


In [17]:
def find_code_column(df: pd.DataFrame) -> str | None:
    """Detect the delay-code column. Common names: 'Code', 'SRT Code', 'Incident Code'."""
    for c in df.columns:
        if str(c).lower() == "code":
            return c
    for c in df.columns:
        if "code" in str(c).lower() and "description" not in str(c).lower():
            return c
    return None


def code_join_check(event_df, lookup_df, label):
    ev_code_col = find_code_column(event_df)
    lk_code_col = find_code_column(lookup_df)
    print(f"\n--- {label} ---")
    print(f"Event code column:  '{ev_code_col}' (dtype: {event_df[ev_code_col].dtype if ev_code_col else 'n/a'})")
    print(f"Lookup code column: '{lk_code_col}' (dtype: {lookup_df[lk_code_col].dtype if lk_code_col else 'n/a'})")
    if ev_code_col is None or lk_code_col is None:
        print("** Could not detect a code column — manual inspection needed. **")
        return
    ev_codes = set(event_df[ev_code_col].dropna().astype(str).unique())
    lk_codes = set(lookup_df[lk_code_col].dropna().astype(str).unique())
    missing = ev_codes - lk_codes
    print(f"Unique codes in events: {len(ev_codes)}")
    print(f"Unique codes in lookup: {len(lk_codes)}")
    print(f"Event codes NOT found in lookup: {len(missing)}")
    if missing:
        print(f"  examples: {sorted(missing)[:10]}")


# We use the most recent full XLSX year (2024) as the representative event file for each mode.
for mode, lookup in [("subway", subway_codes), ("bus", bus_codes), ("streetcar", streetcar_codes)]:
    path = RAW / mode / f"ttc-{mode}-delay-data-2024.xlsx"
    ev = read_excel_stacked(path)
    code_join_check(ev, lookup, f"{mode} 2024 vs Code Descriptions")



--- subway 2024 vs Code Descriptions ---
Event code column:  'Code' (dtype: str)
Lookup code column: 'CODE' (dtype: str)
Unique codes in events: 125
Unique codes in lookup: 140
Event codes NOT found in lookup: 2
  examples: ['PUTO', 'XXXXX']



--- bus 2024 vs Code Descriptions ---
Event code column:  'None' (dtype: n/a)
Lookup code column: 'CODE' (dtype: str)
** Could not detect a code column — manual inspection needed. **



--- streetcar 2024 vs Code Descriptions ---
Event code column:  'None' (dtype: n/a)
Lookup code column: 'CODE' (dtype: str)
** Could not detect a code column — manual inspection needed. **


### 6.3 Date integrity (no Excel serial numbers)

Excel sometimes stores dates as numeric serials (e.g. `44927` for 2023-01-01). If `pd.read_excel` returns numbers instead of `datetime64` for the date column, we've hit that bug and the cleaning step must coerce them explicitly.


In [18]:
for label in [k for k in AUDIT if "codes" not in k]:
    info = AUDIT[label]
    dc = info.get("date_column")
    if dc is None:
        print(f"{label:30s}  no date column detected")
        continue
    print(f"{label:30s}  date column = '{dc}'  min={info.get('date_min')}  max={info.get('date_max')}  unparseable={info.get('date_unparseable')}")


subway_2022                     date column = 'Date'  min=2022-01-01 00:00:00  max=2022-12-31 00:00:00  unparseable=0
subway_2023                     date column = 'Date'  min=2023-01-01 00:00:00  max=2023-12-31 00:00:00  unparseable=0
subway_2024                     date column = 'Date'  min=2024-01-01 00:00:00  max=2024-12-31 00:00:00  unparseable=0
subway_2025plus                 date column = 'Date'  min=2025-01-01 00:00:00  max=2026-01-31 00:00:00  unparseable=0
bus_2022                        date column = 'Date'  min=2022-01-01 00:00:00  max=2022-12-31 00:00:00  unparseable=0
bus_2023                        date column = 'Date'  min=2023-01-01 00:00:00  max=2023-12-31 00:00:00  unparseable=0
bus_2024                        date column = 'Date'  min=2024-01-01 00:00:00  max=2024-12-31 00:00:00  unparseable=0
bus_2025plus                    date column = 'Date'  min=2025-01-01 00:00:00  max=2026-01-31 00:00:00  unparseable=0
streetcar_2022                  date column = 'Date'  mi

## 7. Cross-mode standardisation

The whole point of the audit is to answer: *can these three modes live in one fact table?* This section compares the column sets side-by-side and proposes a mapping.

The table below shows, for every unique column name seen across all 12 event files, which mode contains it. A column present in all three modes is a candidate for the **common** part of the standardised schema; a column present in only one mode becomes a **mode-specific** field that is left null for the other modes.


In [19]:
def union_columns(mode: str) -> set[str]:
    cols = set()
    for k in [f"{mode}_2022", f"{mode}_2023", f"{mode}_2024", f"{mode}_2025plus"]:
        cols.update(c for c in AUDIT[k]["columns"] if c != "_sheet")
    return cols


sub_cols = union_columns("subway")
bus_cols = union_columns("bus")
sc_cols  = union_columns("streetcar")
all_cols = sorted(sub_cols | bus_cols | sc_cols)

cross = pd.DataFrame({
    "subway":    [c in sub_cols for c in all_cols],
    "bus":       [c in bus_cols for c in all_cols],
    "streetcar": [c in sc_cols  for c in all_cols],
}, index=all_cols).replace({True: "x", False: ""})
cross["modes_present"] = (cross == "x").sum(axis=1)
cross = cross.sort_values(["modes_present", cross.index.name or "subway"], ascending=[False, True])
print(cross)


          subway bus streetcar  modes_present
Bound          x   x         x              3
Code           x   x         x              3
Date           x   x         x              3
Day            x   x         x              3
Line           x   x         x              3
Min Delay      x   x         x              3
Min Gap        x   x         x              3
Station        x   x         x              3
Time           x   x         x              3
Vehicle        x   x         x              3
_id            x   x         x              3
Incident           x         x              2
Location           x         x              2
Direction          x                        1
Route              x                        1


**How to read the table:** rows with `x` in all three columns are the universal fields — they form the spine of the combined fact table. Rows present in only one mode become mode-specific extensions.

Even if the *names* differ (e.g. one mode says `Min Delay`, another says `Delay Minutes`), the semantic meaning is the same. The cleaning step will rename to a single canonical column name. The audit's job here is to identify those rename mappings, not to perform them.


## 8. Audit summary

A concise machine-printed summary of the answers to every question in the audit checklist. This is what someone reading the notebook quickly should jump to.


In [20]:
print("=" * 78)
print("RAW DATA AUDIT — SUMMARY")
print("=" * 78)

print("\n[Files and row counts]")
for label in sorted(k for k in AUDIT if "codes" not in k):
    info = AUDIT[label]
    print(f"  {label:25s}  rows={info['rows']:>8,}  cols={info['cols']:>3}  date={info.get('date_min', '?')[:10]} -> {info.get('date_max', '?')[:10]}")

print("\n[Code-description tables]")
for mode in ["subway", "bus", "streetcar"]:
    info = AUDIT[f"{mode}_codes"]
    print(f"  {mode:10s}  rows={info['rows']:>5}  unique_codes={info['unique_codes']:>5}  columns={info['columns']}")

print("\n[Cross-mode column overlap]")
sub_cols = union_columns("subway")
bus_cols = union_columns("bus")
sc_cols  = union_columns("streetcar")
common_all = sub_cols & bus_cols & sc_cols
print(f"  Columns present in ALL three modes ({len(common_all)}): {sorted(common_all)}")
print(f"  Subway-only ({len(sub_cols - bus_cols - sc_cols)}): {sorted(sub_cols - bus_cols - sc_cols)}")
print(f"  Bus-only ({len(bus_cols - sub_cols - sc_cols)}): {sorted(bus_cols - sub_cols - sc_cols)}")
print(f"  Streetcar-only ({len(sc_cols - sub_cols - bus_cols)}): {sorted(sc_cols - sub_cols - bus_cols)}")

print("\n[Total event rows across all years and modes]")
total = sum(AUDIT[k]["rows"] for k in AUDIT if "codes" not in k)
print(f"  {total:,} rows")


RAW DATA AUDIT — SUMMARY

[Files and row counts]
  bus_2022                   rows=  58,707  cols= 11  date=2022-01-01 -> 2022-12-31
  bus_2023                   rows=  56,207  cols= 11  date=2023-01-01 -> 2023-12-31
  bus_2024                   rows=  59,643  cols= 11  date=2024-01-01 -> 2024-12-31
  bus_2025plus               rows=  69,037  cols= 11  date=2025-01-01 -> 2026-01-31
  streetcar_2022             rows=  17,655  cols= 11  date=2022-01-01 -> 2022-12-31
  streetcar_2023             rows=  14,413  cols= 11  date=2023-01-01 -> 2023-12-31
  streetcar_2024             rows=  14,206  cols= 11  date=2024-01-01 -> 2024-12-31
  streetcar_2025plus         rows=  15,785  cols= 11  date=2025-01-01 -> 2026-01-31
  subway_2022                rows=  19,895  cols= 11  date=2022-01-01 -> 2022-12-31
  subway_2023                rows=  22,949  cols= 11  date=2023-01-01 -> 2023-12-31
  subway_2024                rows=  26,467  cols= 11  date=2024-01-01 -> 2024-12-31
  subway_2025plus          

## 9. Findings & next steps

Plain-language interpretation of what the cells above produced. This is the section a recruiter or hiring manager would skim first.

### 9.1 Multi-sheet workbooks? **No.**

Every one of the 12 XLSX files is a **single-sheet** workbook. The sheet name varies (`2022`, `2023`, `Subway`, `Data`), but there is no per-month tab structure. TTC appears to have consolidated to one tab per year.

> The warning in `CLAUDE.md` about monthly-tab workbooks reflected older TTC behaviour. The current files do not need monthly stacking — but our `read_excel_stacked` helper still works correctly (it just stacks one sheet), so the cleaning pipeline can use it safely without special-casing.

### 9.2 Date integrity? **Clean.**

Every date column parsed as `datetime64[us]` (XLSX) or string-then-parseable (CSV), with **zero unparseable values** across all 12 event files. No Excel serial numbers. Date ranges line up exactly with each file's year. The 2025+ CSV runs from 2025-01-01 to 2026-01-31 in all three modes (the feed is current through January 2026).

### 9.3 Delay-code dtype / leading zeros? **No issue.**

Codes are read as strings in both XLSX and CSV. The leading-zero risk is moot anyway because TTC codes are *alphanumeric* (e.g. `MUPAA`, `EUAC`, `MFESA`) rather than zero-padded numeric. We can rely on string equality for joins.

### 9.4 Schema drift — the big surprise

Subway is stable across all four years. **Bus and streetcar are not.** Between 2024 and 2025 the publisher changed how delay reasons are recorded:

| Mode | 2022–2024 columns | 2025+ columns |
|------|-------------------|---------------|
| Subway | `Date, Time, Day, Station, Code, Min Delay, Min Gap, Bound, Line, Vehicle` | same + `_id` |
| Bus | `Date, Route, Time, Day, Location, Incident, Min Delay, Min Gap, Direction, Vehicle` | `Date, Line, Time, Day, Station, Code, Min Delay, Min Gap, Bound, Vehicle, _id` |
| Streetcar | `Date, Line, Time, Day, Location, Incident, Min Delay, Min Gap, Bound, Vehicle` | `Date, Line, Time, Day, Station, Code, Min Delay, Min Gap, Bound, Vehicle, _id` |

What changed for bus and streetcar in 2025:

1. `Route` (bus only) was renamed to `Line` — same meaning.
2. `Location` was renamed to `Station`.
3. `Direction` (bus only) was renamed to `Bound`.
4. **`Incident` (free-text description) was replaced with `Code` (alphanumeric)**, switching from human-readable strings like "Diversion" or "Operations - Operator" to compact codes like `MFESA` that must be joined to `Code Descriptions.csv` for meaning.

This is a real-world ETL problem the cleaning step has to solve. Decision logged in `docs/decisions.md`.

### 9.5 Code-description join feasibility

- **Subway:** 140 codes in the lookup, 125 distinct codes appear in 2024 events. Two event codes are *not* in the lookup: `PUTO` (unknown — investigate during cleaning) and `XXXXX` (clearly a placeholder for missing data — drop or null out).
- **Bus & streetcar 2022–2024:** there is *no* code column in the event files — the `Incident` column already carries the human description, so no join is needed for these years. Our audit helper correctly flagged the absence of a code column.
- **Bus & streetcar 2025+:** the new `Code` column is alphanumeric and joins to `Code Descriptions.csv` (the lookup has 46 codes for bus, 85 for streetcar).

### 9.6 Combined-fact-table feasibility

**Yes — confidently.** After the renames in §9.4 and after handling the `Incident`-vs-`Code` switch by populating a single `delay_description` field (filling from `Incident` directly for 2022–2024 bus/streetcar, joining from the lookup elsewhere), all three modes share these columns:

`event_date, event_time, day_of_week, station/location, line, bound, delay_code, delay_description, delay_minutes, gap_minutes, vehicle, transit_mode, source_year, source_file`

The bus 2022–2024 `Route` column doesn't really need to be mode-specific — it's just the bus's version of `Line` and can be mapped onto `line` in the unified schema. So the standardised fact table is fully symmetric across modes; nothing is *truly* mode-specific in the V1 data model.

### 9.7 Missing-value patterns worth flagging

- `Bound` / `Direction` is missing in 15–37% of rows across all modes and years. This is structural — many delay events (e.g. station-bound incidents, cancellations) genuinely have no associated direction. Treat as legitimately null, not as a data-quality bug.
- `Line` (subway) and `Route` (bus) are missing in <1% of rows — small but non-zero. Worth quantifying in cleaning.
- Subway 2025+ has one row with a missing `Station`; streetcar 2024 has one row with a missing `Location`. Negligible.

### 9.8 Surprises (interview talking points)

1. *"The single biggest surprise was that TTC silently changed the bus and streetcar schemas between 2024 and 2025 — they swapped a free-text incident column for an alphanumeric code that has to be joined against a lookup. Subway was stable; bus and streetcar weren't. I caught it by running a schema-drift comparison year-by-year, before writing any cleaning code."*
2. *"The warning in my own project notes about multi-sheet monthly Excel workbooks turned out to be wrong — TTC had consolidated to one sheet per file. I confirmed that with `openpyxl.load_workbook` in read-only mode before reading the data, so I didn't waste time stacking sheets that didn't exist."*

### 9.9 Next concrete step

Begin Stage 2 (Cleaning). The first cleaning script must:

1. Read each XLSX (single-sheet) and each 2025+ CSV natively.
2. Rename Bus 2022–2024 columns: `Route→line`, `Location→station`, `Direction→bound`, `Incident→delay_description`, `Min Delay→delay_minutes`, `Min Gap→gap_minutes`.
3. Rename Streetcar 2022–2024 columns: `Location→station`, `Incident→delay_description`.
4. For 2025+ bus/streetcar (and all subway years), keep `Code` as `delay_code` and join to that mode's `Code Descriptions.csv` to populate `delay_description`.
5. Drop or null out the `XXXXX` placeholder code; flag `PUTO` for follow-up.
6. Add `transit_mode` and `source_file` columns.
7. Write per-mode standardised CSVs to `data/processed/`.
